In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from abc import ABC, abstractmethod
from typing import List, Callable, Any, Optional
from dataclasses import dataclass
from IPython.display import display
import json
import time
import functools

class AnalisisDataError(Exception): pass
class DataTidakDitemukanError(AnalisisDataError): pass
class FormatDataTidakValidError(AnalisisDataError): pass


In [ ]:
@dataclass(slots=True, frozen=True)
class EmployeeRecord:
    """
    Model data tunggal untuk satu baris karyawan.
    Enkapsulasi: validasi dilakukan di __post_init__ dan
    properti publik dibaca via @property (lihat wrapper di bawah).
    """
    employee_id: str
    work_location: str
    stress_level: str
    mental_health: str
    social_isolation: str
    physical_activity: str
    age: int
    gender: str
    hours_worked: float
    resource_access: str
    sleep_quality: str

    _VALID_STRESS = frozenset(["High", "Medium", "Low", "None"])

    def __post_init__(self) -> None:
        normalized = str(self.stress_level).capitalize()
        if normalized not in self._VALID_STRESS and pd.notna(self.stress_level):
            object.__setattr__(self, "stress_level", "Unspecified") 

        if self.age < 0:
            object.__setattr__(self, "age", 0) 

        if self.hours_worked < 0:
            object.__setattr__(self, "hours_worked", 0.0)  

    @classmethod
    def from_dataframe_row(cls, row: pd.Series) -> "EmployeeRecord":
        """Factory method: buat EmployeeRecord dari satu baris DataFrame."""
        return cls(
            employee_id=str(row.get("employee_id", "N/A")),
            work_location=str(row.get("work_location", "N/A")),
            stress_level=str(row.get("stress_level", "N/A")),
            mental_health=str(row.get("mental_health_condition", "N/A")),
            social_isolation=str(row.get("social_isolation_rating", "N/A")),
            physical_activity=str(row.get("physical_activity", "N/A")),
            age=int(row.get("age", 0)),
            gender=str(row.get("gender", "N/A")),
            hours_worked=float(row.get("hours_worked_per_week", 0.0)),
            resource_access=str(row.get("access_to_mental_health_resources", "N/A")),
            sleep_quality=str(row.get("sleep_quality", "N/A")),
        )

    @staticmethod
    def records_to_dataframe(records: List["EmployeeRecord"]) -> pd.DataFrame:
        """Konversi list EmployeeRecord kembali ke DataFrame untuk analisis."""
        return pd.DataFrame([
            {
                "employee_id": r.employee_id,
                "work_location": r.work_location,
                "stress_level": r.stress_level,
                "mental_health_condition": r.mental_health,
                "social_isolation_rating": r.social_isolation,
                "physical_activity": r.physical_activity,
                "age": r.age,
                "gender": r.gender,
                "hours_worked_per_week": r.hours_worked,
                "access_to_mental_health_resources": r.resource_access,
                "sleep_quality": r.sleep_quality,
            }
            for r in records
        ])

    def __str__(self) -> str:
        return (
            f"EmployeeRecord(id={self.employee_id}, gender={self.gender}, "
            f"age={self.age}, stress={self.stress_level}, "
            f"mental_health={self.mental_health})"
        )


In [ ]:
class PersistenceRepository(ABC):
    @abstractmethod
    def save(self, df: pd.DataFrame, save_name: str) -> None:
        pass

    @abstractmethod
    def load(self, filename: str) -> pd.DataFrame:
        """Muat data kembali dari file. Implementasi opsional di subclass."""
        pass

class JSONRepository(PersistenceRepository):
    def save(self, df: pd.DataFrame, save_name: str) -> None:
        filename = f"{save_name}.json"
        df.to_json(filename, orient="records", indent=4)
        print(f"[SAVED] {filename}")

    def load(self, filename: str) -> pd.DataFrame:
        """Muat DataFrame dari file JSON."""
        return pd.read_json(filename)

class CSVRepository(PersistenceRepository):
    def save(self, df: pd.DataFrame, save_name: str) -> None:
        filename = f"{save_name}.csv"
        df.to_csv(filename, index=False)
        print(f"[SAVED] {filename}")

    def load(self, filename: str) -> pd.DataFrame:
        """Muat DataFrame dari file CSV."""
        return pd.read_csv(filename)


In [ ]:
def execution_logger(func: Callable) -> Callable:
    @functools.wraps(func)
    def wrapper(self: Any, df: pd.DataFrame, *args: Any, **kwargs: Any) -> pd.DataFrame:
        start_time = time.time()
        result = func(self, df, *args, **kwargs)
        duration = time.time() - start_time
        print(f"[LOG] {self.__class__.__name__} selesai dalam {duration:.4f} detik")
        return result
    return wrapper

class PreprocessingStep(ABC):
    @abstractmethod
    def process(self, df: pd.DataFrame) -> pd.DataFrame: pass

class MissingValueHandler(PreprocessingStep):
    @execution_logger
    def process(self, df: pd.DataFrame) -> pd.DataFrame:
        cat_cols = [
            "mental_health_condition", "stress_level", "physical_activity",
            "work_location", "gender", "access_to_mental_health_resources", "sleep_quality",
        ]
        for col in cat_cols:
            if col in df.columns:
                df[col] = df[col].fillna("None")
        num_cols = ["social_isolation_rating", "hours_worked_per_week", "age"]
        for col in num_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
        return df

class ColumnStandardizer(PreprocessingStep):
    @execution_logger
    def process(self, df: pd.DataFrame) -> pd.DataFrame:
        df.columns = [str(col).lower() for col in df.columns]
        return df

class DuplicateRowRemover(PreprocessingStep):
    """Menghapus baris duplikat dari DataFrame."""
    @execution_logger
    def process(self, df: pd.DataFrame) -> pd.DataFrame:
        before = len(df)
        df = df.drop_duplicates()
        removed = before - len(df)
        if removed:
            print(f"[INFO] {removed} baris duplikat dihapus.")
        return df

class EmployeeRecordConverter(PreprocessingStep):
    """
    BARU – Mengkonversi setiap baris DataFrame menjadi EmployeeRecord,
    lalu mengubahnya kembali ke DataFrame yang sudah tervalidasi.
    Ini membuat EmployeeRecord benar-benar digunakan di pipeline.
    """
    @execution_logger
    def process(self, df: pd.DataFrame) -> pd.DataFrame:
        records: List[EmployeeRecord] = [
            EmployeeRecord.from_dataframe_row(row)
            for _, row in df.iterrows()
        ]
        print(f"[INFO] {len(records)} baris dikonversi ke EmployeeRecord.")
        # Demonstrasi: cetak 2 record pertama
        for rec in records[:2]:
            print(f"  → {rec}")
        return EmployeeRecord.records_to_dataframe(records)

class PreprocessingPipeline:
    def __init__(self) -> None:
        self.__steps: List[PreprocessingStep] = []  

    def add_step(self, step: PreprocessingStep) -> None:
        if not isinstance(step, PreprocessingStep):
            raise FormatDataTidakValidError("Step harus berupa PreprocessingStep.")
        self.__steps.append(step)

    @property
    def step_count(self) -> int:
        """Enkapsulasi: jumlah step bisa dibaca tapi tidak bisa diubah langsung."""
        return len(self.__steps)

    def execute(self, df: pd.DataFrame) -> pd.DataFrame:
        if df is None or df.empty:
            raise DataTidakDitemukanError("Dataset kosong.")
        processed_df = df.copy()
        for step in self.__steps:
            processed_df = step.process(processed_df)
        return processed_df


In [ ]:
ANALYSIS_REGISTRY: dict = {}

def register_analysis(name: str):
    """Decorator OCP: daftarkan kelas analisis tanpa mengubah AnalysisFactory."""
    def decorator(cls):
        ANALYSIS_REGISTRY[name] = cls
        return cls
    return decorator

class AbstractAnalysis(ABC):
    @property
    @abstractmethod
    def title(self) -> str: pass

    @abstractmethod
    def run_analysis(self, df: pd.DataFrame) -> pd.DataFrame: pass

    def __repr__(self) -> str:
        return f"<{self.__class__.__name__}(title='{self.title}')>"

class VisualizableMixin(ABC):
    @abstractmethod
    def visualize(self, result_df: pd.DataFrame) -> None: pass

class BaseAnalysis(AbstractAnalysis):
    """
    Kelas induk konkret.
    Enkapsulasi: _description dilindungi dan diekspos via @property.
    """
    def __init__(self, description: str) -> None:
        self._description: str = description   

    @property
    def description(self) -> str:
        """Baca deskripsi analisis."""
        return self._description

    @description.setter
    def description(self, value: str) -> None:
        """Validasi: deskripsi tidak boleh kosong."""
        if not value or not value.strip():
            raise FormatDataTidakValidError("Deskripsi analisis tidak boleh kosong.")
        self._description = value.strip()

    def get_summary(self) -> str:
        """Manfaatkan _description — tampilkan ringkasan kelas."""
        return f"[{self.__class__.__name__}] {self.title} — {self.description}"

    def __str__(self) -> str:
        return self.get_summary()


In [ ]:
@register_analysis("distribusi")
class MentalHealthDistribution(BaseAnalysis, VisualizableMixin):
    def __init__(self) -> None:
        super().__init__("Menghitung distribusi frekuensi kondisi mental seluruh karyawan.")

    @property
    def title(self) -> str:
        return "Distribusi Kondisi Mental Karyawan"

    def run_analysis(self, df: pd.DataFrame) -> pd.DataFrame:
        return df["mental_health_condition"].value_counts().reset_index()

    def visualize(self, result_df: pd.DataFrame) -> None:
        plt.figure(figsize=(8, 5))
        ax = sns.barplot(
            x=result_df.columns[0], y=result_df.columns[1],
            data=result_df, palette="Set2", hue=result_df.columns[0],
        )
        if ax.get_legend() is not None:
            ax.get_legend().remove()
        plt.title(self.title, fontweight="bold")
        plt.tight_layout()
        plt.show()

@register_analysis("lokasi")
class MentalHealthByLocation(BaseAnalysis, VisualizableMixin):
    def __init__(self) -> None:
        super().__init__("Menganalisis kondisi mental karyawan berdasarkan lokasi kerja (Remote/Hybrid/Onsite).")

    @property
    def title(self) -> str:
        return "Kondisi Mental Berdasarkan Lokasi Kerja"

    def run_analysis(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.groupby(["work_location", "mental_health_condition"]).size().reset_index(name="count")

    def visualize(self, result_df: pd.DataFrame) -> None:
        plt.figure(figsize=(10, 5))
        sns.barplot(x="work_location", y="count", hue="mental_health_condition", data=result_df, palette="husl")
        plt.title(self.title, fontweight="bold")
        plt.tight_layout()
        plt.show()

@register_analysis("stres")
class MentalHealthByStressHeatmap(BaseAnalysis, VisualizableMixin):
    def __init__(self) -> None:
        super().__init__("Membuat crosstab antara tingkat stres dan kondisi mental, disajikan sebagai heatmap.")

    @property
    def title(self) -> str:
        return "Heatmap: Tingkat Stres vs Kondisi Mental"

    def run_analysis(self, df: pd.DataFrame) -> pd.DataFrame:
        return pd.crosstab(df["stress_level"], df["mental_health_condition"])

    def visualize(self, result_df: pd.DataFrame) -> None:
        plt.figure(figsize=(8, 5))
        sns.heatmap(result_df, annot=True, fmt="d", cmap="Reds", linewidths=0.5)
        plt.title(self.title, fontweight="bold")
        plt.tight_layout()
        plt.show()

@register_analysis("isolasi")
class MentalHealthByIsolationBoxplot(BaseAnalysis, VisualizableMixin):
    def __init__(self) -> None:
        super().__init__("Membandingkan tingkat isolasi sosial antar kondisi mental via boxplot.")

    @property
    def title(self) -> str:
        return "Tingkat Isolasi Sosial per Kondisi Mental"

    def run_analysis(self, df: pd.DataFrame) -> pd.DataFrame:
        return df[["mental_health_condition", "social_isolation_rating"]]

    def visualize(self, result_df: pd.DataFrame) -> None:
        plt.figure(figsize=(9, 5))
        ax = sns.boxplot(
            x="mental_health_condition", y="social_isolation_rating",
            data=result_df, palette="pastel", hue="mental_health_condition",
        )
        if ax.get_legend() is not None:
            ax.get_legend().remove()
        plt.title(self.title, fontweight="bold")
        plt.tight_layout()
        plt.show()

@register_analysis("aktivitas")
class MentalHealthByPhysicalActivity(BaseAnalysis, VisualizableMixin):
    def __init__(self) -> None:
        super().__init__("Menghitung proporsi kondisi mental berdasarkan frekuensi aktivitas fisik karyawan.")

    @property
    def title(self) -> str:
        return "Proporsi Kondisi Mental Berdasarkan Aktivitas Fisik"

    def run_analysis(self, df: pd.DataFrame) -> pd.DataFrame:
        return pd.crosstab(df["physical_activity"], df["mental_health_condition"], normalize="index") * 100

    def visualize(self, result_df: pd.DataFrame) -> None:
        result_df.plot(kind="barh", stacked=True, figsize=(10, 5), colormap="viridis")
        plt.title(self.title, fontweight="bold")
        plt.legend(title="Kondisi Mental", bbox_to_anchor=(1.05, 1), loc="upper left")
        plt.tight_layout()
        plt.show()

@register_analysis("usia")
class MentalHealthByAgeGroup(BaseAnalysis, VisualizableMixin):
    def __init__(self, bins: Optional[List[int]] = None) -> None:
        """
        Parameter opsional bins membuat kelas ini lebih fleksibel
        (berbeda dari saudara-saudaranya yang all-default) dan
        membuktikan pewarisan lebih bermakna.
        """
        super().__init__("Mengelompokkan karyawan ke bin usia lalu membandingkan kondisi mental tiap kelompok.")
        self._bins = bins or [0, 30, 40, 50, 100]
        self._labels = ["<30 Tahun", "30-39 Tahun", "40-49 Tahun", "≥50 Tahun"]

    @property
    def title(self) -> str:
        return "Kondisi Mental Berdasarkan Kelompok Usia"

    @property
    def age_bins(self) -> List[int]:
        return list(self._bins)

    def run_analysis(self, df: pd.DataFrame) -> pd.DataFrame:
        df_age = df[["mental_health_condition", "age"]].copy()
        df_age = df_age[df_age["age"] > 0]
        df_age["age_group"] = pd.cut(df_age["age"], bins=self._bins, labels=self._labels, right=False)
        return df_age.groupby(["age_group", "mental_health_condition"], observed=False).size().reset_index(name="count")

    def visualize(self, result_df: pd.DataFrame) -> None:
        plt.figure(figsize=(10, 6))
        sns.barplot(x="age_group", y="count", hue="mental_health_condition", data=result_df, palette="Set1")
        plt.title(self.title, fontweight="bold", fontsize=14)
        plt.xlabel("Kelompok Usia")
        plt.ylabel("Jumlah Karyawan")
        plt.legend(title="Kondisi Mental", bbox_to_anchor=(1.05, 1), loc="upper left")
        plt.grid(axis="y", linestyle="--", alpha=0.7)
        plt.tight_layout()
        plt.show()

@register_analysis("gender")
class MentalHealthByGender(BaseAnalysis, VisualizableMixin):
    def __init__(self) -> None:
        super().__init__("Membandingkan distribusi kondisi mental antara kelompok gender.")

    @property
    def title(self) -> str:
        return "Kondisi Mental Berdasarkan Gender"

    def run_analysis(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.groupby(["gender", "mental_health_condition"]).size().reset_index(name="count")

    def visualize(self, result_df: pd.DataFrame) -> None:
        plt.figure(figsize=(10, 5))
        sns.barplot(x="gender", y="count", hue="mental_health_condition", data=result_df, palette="Set1")
        plt.title(self.title, fontweight="bold")
        plt.tight_layout()
        plt.show()

@register_analysis("jam_kerja")
class MentalHealthByWorkingHours(BaseAnalysis, VisualizableMixin):
    def __init__(self) -> None:
        super().__init__("Memeriksa apakah jam kerja mingguan berhubungan dengan kondisi mental karyawan.")

    @property
    def title(self) -> str:
        return "Jam Kerja Mingguan vs Kondisi Mental"

    def run_analysis(self, df: pd.DataFrame) -> pd.DataFrame:
        return df[["mental_health_condition", "hours_worked_per_week"]]

    def visualize(self, result_df: pd.DataFrame) -> None:
        plt.figure(figsize=(10, 5))
        ax = sns.boxplot(
            x="mental_health_condition", y="hours_worked_per_week",
            data=result_df, palette="coolwarm", hue="mental_health_condition",
        )
        if ax.get_legend() is not None:
            ax.get_legend().remove()
        plt.title(self.title, fontweight="bold")
        plt.tight_layout()
        plt.show()

@register_analysis("akses_bantuan")
class MentalHealthByResourcesAccess(BaseAnalysis, VisualizableMixin):
    def __init__(self) -> None:
        super().__init__("Menganalisis apakah akses ke fasilitas kesehatan mental HRD mempengaruhi kondisi karyawan.")

    @property
    def title(self) -> str:
        return "Kondisi Mental Berdasarkan Akses Bantuan HRD"

    def run_analysis(self, df: pd.DataFrame) -> pd.DataFrame:
        return pd.crosstab(df["access_to_mental_health_resources"], df["mental_health_condition"], normalize="index") * 100

    def visualize(self, result_df: pd.DataFrame) -> None:
        result_df.plot(kind="bar", stacked=True, figsize=(9, 5), colormap="Spectral")
        plt.title(self.title, fontweight="bold")
        plt.legend(title="Kondisi Mental", bbox_to_anchor=(1.05, 1), loc="upper left")
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.show()

@register_analysis("tidur")
class MentalHealthBySleepQuality(BaseAnalysis, VisualizableMixin):
    def __init__(self) -> None:
        super().__init__("Mengevaluasi hubungan antara kualitas tidur karyawan dan kondisi mental mereka.")

    @property
    def title(self) -> str:
        return "Kondisi Mental Berdasarkan Kualitas Tidur"

    def run_analysis(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.groupby(["sleep_quality", "mental_health_condition"]).size().reset_index(name="count")

    def visualize(self, result_df: pd.DataFrame) -> None:
        plt.figure(figsize=(10, 5))
        order = ["Poor", "Average", "Good"]
        sns.barplot(x="sleep_quality", y="count", hue="mental_health_condition", data=result_df, order=order, palette="magma")
        plt.title(self.title, fontweight="bold")
        plt.tight_layout()
        plt.show()

@register_analysis("ringkasan")
class SummaryStatisticsAnalysis(BaseAnalysis):
    """
    Bukti modularitas VisualizableMixin:
    kelas ini TIDAK mengimplementasikan VisualizableMixin karena
    hasilnya cukup ditampilkan sebagai tabel, bukan grafik.
    Sesuai ISP — klien tidak dipaksa bergantung pada interface yang tidak dipakai.
    """
    def __init__(self) -> None:
        super().__init__("Menghasilkan statistik deskriptif numerik dataset (usia, jam kerja, isolasi sosial).")

    @property
    def title(self) -> str:
        return "Ringkasan Statistik Deskriptif Dataset"

    def run_analysis(self, df: pd.DataFrame) -> pd.DataFrame:
        num_cols = ["age", "hours_worked_per_week", "social_isolation_rating"]
        existing = [c for c in num_cols if c in df.columns]
        return df[existing].describe().T


In [ ]:
class AnalysisFactory:
    @staticmethod
    def create_analysis(type_name: str) -> BaseAnalysis:
        if type_name in ANALYSIS_REGISTRY:
            return ANALYSIS_REGISTRY[type_name]()
        raise FormatDataTidakValidError(
            f"Tipe '{type_name}' tidak valid. Tersedia: {list(ANALYSIS_REGISTRY.keys())}"
        )

class AnalysisRunner:
    """
    Enkapsulasi: _analysis dan _repositories private, dibaca via @property.
    DIP  : bergantung pada AbstractAnalysis & PersistenceRepository (abstraksi).
    """
    def __init__(
        self,
        analysis: AbstractAnalysis,
        repositories: List[PersistenceRepository],
    ) -> None:
        self._analysis = analysis
        self._repositories = repositories

    @property
    def analysis(self) -> AbstractAnalysis:
        return self._analysis

    @analysis.setter
    def analysis(self, value: AbstractAnalysis) -> None:
        if not isinstance(value, AbstractAnalysis):
            raise FormatDataTidakValidError("analysis harus berupa AbstractAnalysis.")
        self._analysis = value

    @property
    def repository_count(self) -> int:
        return len(self._repositories)

    def run_and_save(self, df: pd.DataFrame, save_name: str) -> pd.DataFrame:
        result = self._analysis.run_analysis(df)
        save_df = result.reset_index() if not isinstance(result.index, pd.RangeIndex) else result
        for repo in self._repositories:
            repo.save(save_df, save_name)
        return result


In [ ]:
try:
    raw_df = pd.read_csv("Impact_of_Remote_Work_on_Mental_Health.csv")
    pipeline = PreprocessingPipeline()
    pipeline.add_step(ColumnStandardizer())
    pipeline.add_step(MissingValueHandler())
    pipeline.add_step(DuplicateRowRemover())      
    pipeline.add_step(EmployeeRecordConverter())  
    df_clean = pipeline.execute(raw_df)

    print(f"\n✅ Pipeline selesai ({pipeline.step_count} langkah).")
    print(f"   Baris siap dianalisis : {df_clean.shape[0]}")
    print(f"   Kolom                 : {list(df_clean.columns)}")

    repo_list = [JSONRepository(), CSVRepository()]

except Exception as e:
    print(f"❌ Gagal memuat data: {e}")


In [ ]:
print("=== Demo LSP: semua subclass mengembalikan DataFrame ===\n")
lsp_candidates = ["distribusi", "ringkasan", "usia"]
for name in lsp_candidates:
    obj: AbstractAnalysis = AnalysisFactory.create_analysis(name)  
    result = obj.run_analysis(df_clean)
    assert isinstance(result, pd.DataFrame), f"{name} melanggar LSP!"
    print(f"  ✓ {obj.title} → {result.shape} — LSP OK")
    if isinstance(obj, BaseAnalysis):
        print(f"    └─ description: {obj.description}")

print("\nSemua subclass lolos uji LSP.")

print("\n=== Demo ISP: membuktikan SummaryStatisticsAnalysis tidak punya visualize ===\n")
assert not hasattr(AnalysisFactory.create_analysis("ringkasan"), "visualize"), \
    "ISP violated! SummaryStatisticsAnalysis seharusnya tidak punya method visualize."
print("  ✓ SummaryStatisticsAnalysis tidak memiliki method visualize — ISP terbukti via kode!")
print("  ✓ VisualizableMixin hanya diimplementasikan oleh kelas yang membutuhkan visualisasi.")

In [ ]:
print("=== Demo @property pada BaseAnalysis ===\n")

obj = AnalysisFactory.create_analysis("stres")
print(f"Sebelum : {obj.description}")
obj.description = "Deskripsi baru: heatmap korelasi stres dan kesehatan mental."
print(f"Sesudah : {obj.description}")

try:
    obj.description = ""
except FormatDataTidakValidError as e:
    print(f"Validasi ✓ — setter menolak string kosong (FormatDataTidakValidError): {e}")

print()
print("=== Demo __str__ pada BaseAnalysis ===\n")
for name in ["distribusi", "lokasi", "ringkasan"]:
    print(" ", str(AnalysisFactory.create_analysis(name)))

print()
print("=== Demo @property pada AnalysisRunner ===\n")
runner = AnalysisRunner(
    AnalysisFactory.create_analysis("distribusi"),
    repo_list,
)
print(f"Jumlah repositori : {runner.repository_count}")
print(f"Analisis aktif    : {runner.analysis}")

try:
    runner.analysis = "bukan_objek"
except FormatDataTidakValidError as e:
    print(f"Validasi ✓ — setter menolak non-AbstractAnalysis: {e}")


In [ ]:
try:
    analisis_obj = AnalysisFactory.create_analysis("distribusi")
    runner = AnalysisRunner(analisis_obj, repositories=repo_list)
    hasil = runner.run_and_save(df_clean, "hasil_distribusi")
    print(str(analisis_obj))
    display(hasil.head())
    analisis_obj.visualize(hasil)
except Exception as e:
    print(e)


In [ ]:
try:
    analisis_obj = AnalysisFactory.create_analysis("lokasi")
    runner = AnalysisRunner(analisis_obj, repositories=repo_list)
    hasil = runner.run_and_save(df_clean, "hasil_lokasi")
    print(str(analisis_obj))
    display(hasil.head(6))
    analisis_obj.visualize(hasil)
except Exception as e:
    print(e)


In [ ]:
try:
    analisis_obj = AnalysisFactory.create_analysis("stres")
    runner = AnalysisRunner(analisis_obj, repositories=repo_list)
    hasil = runner.run_and_save(df_clean, "hasil_stres")
    print(str(analisis_obj))
    display(hasil.head())
    analisis_obj.visualize(hasil)
except Exception as e:
    print(e)


In [ ]:
try:
    analisis_obj = AnalysisFactory.create_analysis("isolasi")
    runner = AnalysisRunner(analisis_obj, repositories=repo_list)
    hasil = runner.run_and_save(df_clean, "hasil_isolasi")
    print(str(analisis_obj))
    display(hasil.head(3))
    analisis_obj.visualize(hasil)
except Exception as e:
    print(e)


In [ ]:
try:
    analisis_obj = AnalysisFactory.create_analysis("aktivitas")
    runner = AnalysisRunner(analisis_obj, repositories=repo_list)
    hasil = runner.run_and_save(df_clean, "hasil_aktivitas")
    print(str(analisis_obj))
    display(hasil.head())
    analisis_obj.visualize(hasil)
except Exception as e:
    print(e)


In [ ]:
try:
    analisis_obj = AnalysisFactory.create_analysis("usia")
    runner = AnalysisRunner(analisis_obj, repositories=repo_list)
    hasil = runner.run_and_save(df_clean, "hasil_usia")
    print(str(analisis_obj))
    display(hasil.head(3))
    analisis_obj.visualize(hasil)
except Exception as e:
    print(e)


In [ ]:
try:
    analisis_obj = AnalysisFactory.create_analysis("gender")
    runner = AnalysisRunner(analisis_obj, repositories=repo_list)
    hasil = runner.run_and_save(df_clean, "hasil_gender")
    print(str(analisis_obj))
    display(hasil.head(6))
    analisis_obj.visualize(hasil)
except Exception as e:
    print(e)


In [ ]:
try:
    analisis_obj = AnalysisFactory.create_analysis("jam_kerja")
    runner = AnalysisRunner(analisis_obj, repositories=repo_list)
    hasil = runner.run_and_save(df_clean, "hasil_jam_kerja")
    print(str(analisis_obj))
    display(hasil.head(3))
    analisis_obj.visualize(hasil)
except Exception as e:
    print(e)


In [ ]:
try:
    analisis_obj = AnalysisFactory.create_analysis("akses_bantuan")
    runner = AnalysisRunner(analisis_obj, repositories=repo_list)
    hasil = runner.run_and_save(df_clean, "hasil_akses_bantuan")
    print(str(analisis_obj))
    display(hasil.head())
    analisis_obj.visualize(hasil)
except Exception as e:
    print(e)


In [ ]:
try:
    analisis_obj = AnalysisFactory.create_analysis("tidur")
    runner = AnalysisRunner(analisis_obj, repositories=repo_list)
    hasil = runner.run_and_save(df_clean, "hasil_tidur")
    print(str(analisis_obj))
    display(hasil.head())
    analisis_obj.visualize(hasil)
except Exception as e:
    print(e)


In [ ]:
try:
    analisis_obj = AnalysisFactory.create_analysis("ringkasan")
    runner = AnalysisRunner(analisis_obj, repositories=repo_list)
    hasil = runner.run_and_save(df_clean, "hasil_ringkasan")
    print(str(analisis_obj))
    display(hasil.head())
except Exception as e:
    print(e)
